# Retrieval Lab: 日本語 BM25 tutorial



## Goal

ローカルの日本語 corpus と qrels を使い、BM25 の Recall を確認します。
この notebook はネットワーク、実モデル、秘密情報を使いません。


## Setup

入力は `examples/japanese/corpus` と `examples/japanese/qrels.jsonl` です。
実行時の current working directory はリポジトリルートを想定し、
見つからない場合は親ディレクトリを探します。文書 relevance を評価します。


## Steps

### 1. Load the local inputs


In [ ]:
from pathlib import Path

from retrieval_lab import EvaluationDataset, EvaluationRunner, load_documents

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "src" / "retrieval_lab").is_dir()
    ),
    None,
)
assert project_root is not None, "run from the repository or a child directory"
corpus_path = project_root / "examples" / "japanese" / "corpus"
qrels_path = project_root / "examples" / "japanese" / "qrels.jsonl"
assert corpus_path.is_dir() and qrels_path.is_file()

In [ ]:
documents = load_documents(corpus_path)
dataset = EvaluationDataset.from_jsonl(qrels_path)
print(f"documents={len(documents)}, queries={len(dataset.queries)}")

### 2. Run BM25

すべての query を一度検索し、top_k は 1 と 3 に固定します。


## Checks

結果の retriever 名、query 数、Recall の範囲を小さな assert で確認します。


In [ ]:
result = EvaluationRunner.from_dataset(
    documents=documents,
    dataset=dataset,
    strategies=("bm25",),
    top_k=(1, 3),
    seed=42,
).run()
metrics_preview = {
    f"Recall@{cutoff}": round(result.metrics["bm25"].recall_at(cutoff), 3)
    for cutoff in (1, 3)
}
print(metrics_preview)

In [ ]:
assert result.manifest["retrievers"] == ["bm25"]
assert len(result.query_results["bm25"]) == 4
assert all(0.0 <= value <= 1.0 for value in metrics_preview.values())
print("checks=ok")

## Next Steps

Dense/Hybrid、Callable Retriever、precomputed ranking の例は
`examples/` と `docs/tutorial.md` を参照してください。
